In [ ]:
import pathlib
import pandas as pd
import pathlib
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import os
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence

import cebra
from cebra import CEBRA

import tempfile
from pathlib import Path

import pickle
from pathlib import Path


import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (
    Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization,
    LSTM, Bidirectional, GlobalAveragePooling1D
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
import tensorflow as tf
import polars as pl
from sklearn.metrics import confusion_matrix

import numpy as np
from sktime.transformations.panel.rocket import MiniRocketMultivariate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeClassifierCV
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, f1_score

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

In [ ]:
# https://www.kaggle.com/code/ahnc2171/cmi-nnn

DATA_DIR = pathlib.Path('/Users/ashhadulislam/projects/general_data/CMI/ Detect Behavior with Sensor Data/cmi-detect-behavior-with-sensor-data/')  

train_path      = os.path.join(DATA_DIR, "train.csv")
test_path       = os.path.join(DATA_DIR, "test.csv")          #
train_demo_path = os.path.join(DATA_DIR, "train_demographics.csv")
test_demo_path  = os.path.join(DATA_DIR, "test_demographics.csv")

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)
train_demo = pd.read_csv(train_demo_path)
test_demo  = pd.read_csv(test_demo_path)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("Train demo :", train_demo.shape)
train_df = train_df.merge(train_demo, on="subject", how="left")

TOF_COLS = [c for c in train_df.columns if c.startswith("tof_")]
THM_COLS = [c for c in train_df.columns if c.startswith("thm_")]

print("Num ToF cols:", len(TOF_COLS))
print("Num Thermal cols:", len(THM_COLS))


IMU_BASE_COLS = [
    "acc_x","acc_y","acc_z",
    "rot_w","rot_x","rot_y","rot_z"
]

IMU_FEATURE_NAMES = [
    "acc_x", "acc_y", "acc_z",
    "rot_6d_0", "rot_6d_1", "rot_6d_2",
    "rot_6d_3", "rot_6d_4", "rot_6d_5",
    "angular_vel_x", "angular_vel_y", "angular_vel_z",
    "linear_acc_x", "linear_acc_y", "linear_acc_z",
    "jerk_x", "jerk_y", "jerk_z",
    "acc_mag", "jerk_mag",
]

def normalize_quaternion(quat: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(quat, axis=-1, keepdims=True)
    norm = np.where(norm > 1e-8, norm, 1.0)
    return quat / norm

def quaternion_to_6d_rotation(quat: np.ndarray) -> np.ndarray:
    if quat.ndim == 1:
        quat = quat.reshape(1, -1)

    has_nan = np.any(np.isnan(quat), axis=-1)
    result = np.full((*quat.shape[:-1], 6), np.nan, dtype=np.float32)

    valid_mask = ~has_nan & ~np.all(np.isclose(quat, 0), axis=-1)
    if not np.any(valid_mask):
        return result

    valid_quat = quat[valid_mask]
    try:
        valid_quat_norm = normalize_quaternion(valid_quat)
        rotations = R.from_quat(valid_quat_norm)
        rotation_matrices = rotations.as_matrix()
        result[valid_mask] = rotation_matrices[:, :, :2].reshape(-1, 6)
    except (ValueError, RuntimeError):
        pass
    return result.astype(np.float32)

def remove_gravity_from_acc(
    acc_data: np.ndarray,
    rot_data: np.ndarray,
    gravity_world: np.ndarray = np.array([0, 0, 9.81], dtype=np.float32),
) -> np.ndarray:
    num_samples = acc_data.shape[0]
    linear_accel = np.full_like(acc_data, np.nan, dtype=np.float32)

    for i in range(num_samples):
        acc_i = acc_data[i]
        quat_i = rot_data[i]

        if np.any(np.isnan(acc_i)) or np.any(np.isnan(quat_i)):
            continue
        if np.all(np.isclose(quat_i, 0)):
            linear_accel[i] = acc_i
            continue

        try:
            quat_norm = normalize_quaternion(quat_i.reshape(1, -1))[0]
            rot = R.from_quat(quat_norm)
            gravity_sensor = rot.apply(gravity_world, inverse=True)
            linear_accel[i] = acc_i - gravity_sensor
        except (ValueError, RuntimeError):
            continue

    return linear_accel

def calculate_angular_velocity_from_quat(
    rot_data: np.ndarray,
    time_delta: float = 1/100.0,
) -> np.ndarray:
    num_samples = rot_data.shape[0]
    angular_vel = np.full((num_samples, 3), np.nan, dtype=np.float32)

    for i in range(num_samples - 1):
        q_t = rot_data[i]
        q_tp = rot_data[i + 1]

        if np.any(np.isnan(q_t)) or np.any(np.isnan(q_tp)):
            continue
        if np.all(np.isclose(q_t, 0)) or np.all(np.isclose(q_tp, 0)):
            continue
        try:
            q_t_norm  = normalize_quaternion(q_t.reshape(1, -1))[0]
            q_tp_norm = normalize_quaternion(q_tp.reshape(1, -1))[0]
            rot_t  = R.from_quat(q_t_norm)
            rot_tp = R.from_quat(q_tp_norm)
            delta_rot = rot_t.inv() * rot_tp
            angular_vel[i] = delta_rot.as_rotvec() / time_delta
        except (ValueError, RuntimeError):
            continue

    return angular_vel

def make_imu_features_from_np(data: np.ndarray) -> np.ndarray:
    """
    data: (T, 8) = [acc_x,acc_y,acc_z, rot_x,rot_y,rot_z,rot_w, handedness]
    """
    acc = data[:, :3].copy()
    rot = data[:, 3:7].copy()
    handedness = data[0, 7]

    feat = acc.copy()

    rot_6d = quaternion_to_6d_rotation(rot)
    feat = np.concatenate([feat, rot_6d], axis=1)

    angular_velocity = calculate_angular_velocity_from_quat(rot)
    feat = np.concatenate([feat, angular_velocity], axis=1)

    linear_acc = remove_gravity_from_acc(acc, rot)
    feat = np.concatenate([feat, linear_acc], axis=1)

    feat = np.nan_to_num(feat, nan=0.0).astype(np.float32)

    if handedness == 0:
        feat[:, 0] *= -1.0
        feat[:, 3] *= -1.0
        feat[:, 7] *= -1.0
        feat[:, 9]  *= -1.0
        feat[:, 10] *= -1.0
        feat[:, 11] *= -1.0
        feat[:, 12] *= -1.0
        feat[:, 13] *= -1.0

    lin = feat[:, -3:].astype(np.float32)
    jerk = np.zeros_like(lin, dtype=np.float32)
    if len(lin) > 1:
        jerk[1:] = lin[1:] - lin[:-1]

    acc_mag  = np.linalg.norm(lin, axis=1)
    jerk_mag = np.linalg.norm(jerk, axis=1)

    feat = np.concatenate(
        [
            feat,
            jerk,
            acc_mag[:, None],
            jerk_mag[:, None],
        ],
        axis=1,
    )

    return feat.astype(np.float32)

def preprocess_sequence_multisensor(grp: pd.DataFrame) -> pd.DataFrame:
    grp = grp.sort_values("sequence_counter").reset_index(drop=True)

    grp[IMU_BASE_COLS] = grp[IMU_BASE_COLS].ffill().bfill()

    acc_np = grp[["acc_x","acc_y","acc_z"]].values.astype(np.float32)
    rot_np = grp[["rot_x","rot_y","rot_z","rot_w"]].values.astype(np.float32)
    handed = grp["handedness"].fillna(1).values.astype(np.float32)

    imu_data_np = np.concatenate([acc_np, rot_np, handed.reshape(-1,1)], axis=1)
    imu_feat_np = make_imu_features_from_np(imu_data_np)

    for i, c in enumerate(IMU_FEATURE_NAMES):
        grp[c] = imu_feat_np[:, i]

    if len(TOF_COLS) > 0:
        tof_vals = grp[TOF_COLS].values.astype(np.float32)
        max_val = 254.0
        mask_neg1 = (tof_vals == -1)
        tof_vals[mask_neg1] = max_val
        tof_vals = tof_vals / max_val
        tof_vals = np.nan_to_num(tof_vals, nan=0.0)
        grp[TOF_COLS] = tof_vals

    if len(THM_COLS) > 0:
        grp[THM_COLS] = grp[THM_COLS].ffill().bfill()
        grp[THM_COLS] = grp[THM_COLS].fillna(method="ffill").fillna(method="bfill").fillna(0)

    eps = 1e-6

    imu_cols = IMU_FEATURE_NAMES
    for c in imu_cols:
        mu  = grp[c].mean()
        std = grp[c].std()
        if std < eps:
            std = eps
        grp[c] = (grp[c] - mu) / std

    if len(TOF_COLS) > 0:
        for c in TOF_COLS:
            mu  = grp[c].mean()
            std = grp[c].std()
            if std < eps:
                std = eps
            grp[c] = (grp[c] - mu) / std

    if len(THM_COLS) > 0:
        for c in THM_COLS:
            mu  = grp[c].mean()
            std = grp[c].std()
            if std < eps:
                std = eps
            grp[c] = (grp[c] - mu) / std

    grp[imu_cols] = grp[imu_cols].ffill().bfill().fillna(0)
    if len(TOF_COLS) > 0:
        grp[TOF_COLS] = grp[TOF_COLS].ffill().bfill().fillna(0)
    if len(THM_COLS) > 0:
        grp[THM_COLS] = grp[THM_COLS].ffill().bfill().fillna(0)

    return grp

train_df=pd.read_csv('data/extractedPhysicsFeatures_train.csv')




In [ ]:

TARGET_GESTURES = [
    'Above ear - pull hair',
    'Cheek - pinch skin',
    'Eyebrow - pull hair',
    'Eyelash - pull hair',
    'Forehead - pull hairline',
    'Forehead - scratch',
    'Neck - pinch skin',
    'Neck - scratch',
]

NON_TARGET_GESTURES = [
    'Write name on leg',
    'Wave hello',
    'Glasses on/off',
    'Text on phone',
    'Write name in air',
    'Feel around in tray and pull out an object',
    'Scratch knee/leg skin',
    'Pull air toward your face',
    'Drink from bottle/cup',
    'Pinch knee/leg skin',
]

ALL_GESTURES_SET = TARGET_GESTURES + NON_TARGET_GESTURES


## below code takes quite a while so we have saved the output dataframe

'''
train_df = train_df.groupby("sequence_id", group_keys=False).apply(preprocess_sequence_multisensor)
print("Train after preprocess:", train_df.shape)
'''


#gestures = sorted(df["gesture"].unique())
gesture2idx = {g: i for i, g in enumerate(ALL_GESTURES_SET)}
idx2gesture = {i: g for g, i in gesture2idx.items()}
num_classes = len(ALL_GESTURES_SET)
print("Num gesture classes:", num_classes)


# ------------------------------------------------------------------
# 4.  SEQUENCE BUILDING HELPERS
# ------------------------------------------------------------------

def preprocess_sequence(df_seq: pd.DataFrame, feature_columns: list[str]) -> np.ndarray:
    """Fill→scale a *single* sequence dataframe and return float32 numpy."""
    data = df_seq[feature_columns].copy()
    data = data.ffill().bfill().fillna(0.0)
    #print(data)
    scaled = StandardScaler().fit_transform(data)   # per-sequence scaler (unchanged)
    #print(scaled)
    return scaled.astype("float32")
    #return data.values




In [ ]:
gesture2idx

<img src="arch.png" width="400">

In [ ]:
# here we define the experiment scenarios



drop_thermal_and_tof_options=[True,False]
use_raw_imus_options=[False,True]
use_physics_features_options=[False,True]
cebra_output_dims_list=[2,8,20,32]

results=[]
splits_store = {}

for cebra_output_dims in cebra_output_dims_list:
    for drop_thermal_and_tof in drop_thermal_and_tof_options:
        for use_raw_imus in use_raw_imus_options:
            for use_physics_features in use_physics_features_options:
                if drop_thermal_and_tof and not use_physics_features and not use_raw_imus:
                    continue
                
                df=train_df.copy()
                df["y_multi"] = df["gesture"].map(gesture2idx).astype(int)
                y_bin=[]
                for v in df["gesture"]:
                    if v in TARGET_GESTURES:
                        y_bin.append(1)
                    else:
                        y_bin.append(0)
                df["y_bin"]=y_bin
                print(Counter(y_bin))
                thermal_tof_cols = [c for c in df.columns if c.startswith(("thm_", "tof_"))]                

                excluded_cols = {
                    "gesture", "sequence_type", "behavior", "orientation",  # train-only targets
                    "row_id", "subject", "phase",                            # meta
                    "sequence_id", "sequence_counter",                         # ids
                    "y_multi","y_bin"                                               # class label for multiclass
                }

                if drop_thermal_and_tof:
                    excluded_cols.update(thermal_tof_cols)
                    #print(f"Ignoring {len(thermal_tof_cols)} thermopile/TOF channels → set drop_thermal_and_tof=False to use them.")
                if not use_raw_imus:
                    excluded_cols.update(IMU_BASE_COLS)
                    #print(f"Ignoring {len(IMU_BASE_COLS)} IMU_BASE_COLS channels → set use_raw_imus_options=True to use them.")
                if not use_physics_features:
                    excluded_cols.update(IMU_FEATURE_NAMES)
                    #print(f"Ignoring {len(IMU_FEATURE_NAMES)} IMU_FEATURE_NAMES channels → set use_physics_features=True to use them.")


                # --- NEW: demographic numeric columns --------------------------------
                demographic_cols = [
                    "adult_child", "age", "sex", "handedness",
                    "height_cm", "shoulder_to_wrist_cm", "elbow_to_wrist_cm",
                ]

                # Combine sensor + demographic feature list
                feature_cols = [c for c in df.columns if c not in excluded_cols]
                #print(f"Using {len(feature_cols)} feature columns for training, including demographics:")
                #print(sorted(feature_cols)[:15], "…")

                # Check missing values
                nan_total = df[feature_cols].isna().sum().sum()
                #print(f"Total NaNs inside feature matrix: {nan_total:,}")



                print("Constructing padded tensor dataset …")
                seq_groups = df.groupby("sequence_id")

                X, seq_lengths = [], []
                for i, (_, seq) in enumerate(seq_groups):
                    if i and i % 500 == 0:
                        print(f"  processed {i} sequences …")
                    arr = preprocess_sequence(seq, feature_cols)
                    X.append(arr)
                    seq_lengths.append(arr.shape[0])

                pad_len = int(np.percentile(seq_lengths, 90))
                print(f"90th-percentile length = {pad_len} → fixed pad length chosen")



                X = pad_sequences(X, maxlen=pad_len, dtype="float32", padding="post", truncating="post")

                y_multi = seq_groups["y_multi"].first().values
                y_bin = seq_groups["y_bin"].first().values


                # ------------------------------------------------------------------
                # 5.  TRAIN/VAL SPLIT & MODEL
                # ------------------------------------------------------------------
                X_train, X_val, y_train, y_val = train_test_split(
                    X, y_bin, test_size=0.20, random_state=42, stratify=y_bin
                )


                scenario_key = (
                    f"dtt={int(drop_thermal_and_tof)}_"
                    f"uri={int(use_raw_imus)}_"
                    f"uff={int(use_physics_features)}_"
                    f"ced={cebra_output_dims}"
                )

                splits_store[scenario_key] = {
                    "X_train": X_train,
                    "y_train": y_train,
                    "X_val": X_val,
                    "y_val": y_val,
                }

                Path("res").mkdir(exist_ok=True)

                with open("res/all_splits.pkl", "wb") as f:
                    pickle.dump(splits_store, f)

                print("Saved splits for", len(splits_store), "scenarios → res/all_splits.pkl")                

                print(X.shape,y_bin.shape,X_train.shape,y_train.shape,X_val.shape,y_val.shape)
                print(np.unique(y_train),np.unique(y_val))
                print(Counter(y_train),Counter(y_val),Counter(y_bin))
                assert X_train.shape[2]==len(feature_cols)

                # Flatten trials into one long time series
                X_train_flat = X_train.reshape(-1, X_train.shape[-1])
                X_val_flat   = X_val.reshape(-1, X_val.shape[-1])

                print(X_train_flat.shape)
                print(X_val_flat.shape)



                cebra_model = CEBRA(
                    model_architecture="offset10-model",
                    conditional="time",      # 🔑 UNSUPERVISED
                    time_offsets=10,         # ~10% of 103
                    output_dimension=cebra_output_dims,
                    batch_size=512,
                    max_iterations=10000,
                    device="cuda_if_available",
                    verbose=True,
                )

                cebra_model.fit(X_train_flat)

                cebra_model_name=f'cebra_dtt_{int(drop_thermal_and_tof)}_uri_{int(use_raw_imus)}_uff_{int(use_physics_features)}'
                cebra_model_name=cebra_model_name+f'_ced_{cebra_output_dims}.pt'
                cebra_model_path =  f"models/{cebra_model_name}"                

                cebra_model.save(cebra_model_path)
                

                res_dic={
                    'drop_thermal_and_tof_options':drop_thermal_and_tof,
                    'use_raw_imus':use_raw_imus,
                    'use_physics_features':use_physics_features,
                    'num_features':len(feature_cols),
                    'cebra_output_dims':cebra_output_dims,
                    'train_dist':Counter(y_train),
                    'val_dist':Counter(y_val),
                    'data_dist':Counter(y_bin),
                    'cebra_model_path':cebra_model_path

                    }
                results.append(res_dic)
                print(res_dic)
                results_df=pd.DataFrame(results)
                results_df.to_csv('res/cebra_feats.csv',index=False)





In [ ]:
len(splits_store.keys())